# Level 1 — Task 3: K-Nearest Neighbors (KNN) Classifier
**Dataset:** Iris | **Tools:** pandas, scikit-learn, matplotlib, seaborn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid")


## 2. Load & Prepare Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload iris.csv

df = pd.read_csv("iris.csv")

le = LabelEncoder()
df["species_encoded"] = le.fit_transform(df["species"])

X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species_encoded"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Classes: {list(le.classes_)}")


## 3. Find Optimal K

In [ ]:
k_range = range(1, 31)
train_scores, test_scores, cv_scores = [], [], []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))
    cv = cross_val_score(knn, X_scaled, y, cv=5, scoring="accuracy")
    cv_scores.append(cv.mean())

optimal_k = k_range[np.argmax(cv_scores)]
print(f"Optimal K (by 5-fold CV accuracy): {optimal_k}")


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(k_range, train_scores, label="Train Accuracy", marker="o", markersize=4)
plt.plot(k_range, test_scores,  label="Test Accuracy",  marker="s", markersize=4)
plt.plot(k_range, cv_scores,    label="CV Accuracy (5-fold)", marker="^", markersize=4, linestyle="--")
plt.axvline(optimal_k, color="red", linestyle=":", linewidth=1.5, label=f"Optimal K={optimal_k}")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy vs K Value")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Train Final Model

In [ ]:
knn = KNeighborsClassifier(n_neighbors=optimal_k)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))


## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Confusion Matrix (K={optimal_k})")

sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt=".2%",
            cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title("Normalized Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 6. Compare K Values

In [ ]:
compare_k = [1, 3, 5, 7, optimal_k, 15, 21]
rows = []
for k in compare_k:
    knn_tmp = KNeighborsClassifier(n_neighbors=k)
    knn_tmp.fit(X_train, y_train)
    cv = cross_val_score(knn_tmp, X_scaled, y, cv=5)
    rows.append({
        "K": k,
        "Train Acc": knn_tmp.score(X_train, y_train),
        "Test Acc": knn_tmp.score(X_test, y_test),
        "CV Mean": cv.mean(),
        "CV Std": cv.std()
    })

results_df = pd.DataFrame(rows).set_index("K")
print(results_df.round(4))


In [ ]:
results_df[["Train Acc", "Test Acc", "CV Mean"]].plot(
    kind="bar", figsize=(10, 5), edgecolor="white"
)
plt.title("Model Performance Comparison Across K Values")
plt.xlabel("K")
plt.ylabel("Accuracy")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 7. Summary

| Metric | Value |
|---|---|
| Optimal K | See output above |
| Test Accuracy | See output above |
| CV Accuracy | See output above |

Low K → low bias, high variance (overfitting risk). High K → high bias, low variance (underfitting risk). The optimal K balances both via cross-validation.